<a href="https://colab.research.google.com/github/TejashwiniByrappa/Tejashwini-landing-page/blob/main/heart_disease_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
# 1. Install required libraries (runs quietly)
!pip install -q transformers torch accelerate pandas

import torch
import pandas as pd
from transformers import pipeline

# 2. Define the exact file path
file_path = "/content/heart.csv"

# Load the dataset
df = pd.read_csv(file_path)
print(f"Successfully loaded '{file_path}' with {len(df)} rows.")

# 3. Load Open-Source LLM (No API Key Required)
print("\nLoading local LLM into memory... (This takes ~30 seconds)")
device = "cuda" if torch.cuda.is_available() else "cpu"

pipe = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-1.5B-Instruct",
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map="auto" if device == "cuda" else None
)

# 4. Select a patient row to evaluate (Row 0 is the first patient)
patient_index = 0
sample_patient = df.iloc[patient_index].to_dict()

# 5. Build prompt with patient details
messages = [
    {
        "role": "system",
        "content": "You are a clinical AI assistant analyzing patient risk factors."
    },
    {
        "role": "user",
        "content": f"""Analyze this patient profile and determine whether they are at high risk for heart disease. Explain your rationale concisely based on the numbers provided.

Patient Profile Data:
{sample_patient}

Assessment:"""
    }
]

# 6. Generate LLM Analysis
print("\n--- LLM Evaluation Output ---")
outputs = pipe(messages, max_new_tokens=256, do_sample=False)
print(outputs[0]["generated_text"][-1]["content"])

Successfully loaded '/content/heart.csv' with 918 rows.

Loading local LLM into memory... (This takes ~30 seconds)


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

...

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.



--- LLM Evaluation Output ---


[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Based on the patient's profile data:

- Age: 40 (within normal range)
- Sex: Male (no specific risk factor)
- Chest Pain Type: Atypical Angina (ATA) (low risk factor)
- Resting Blood Pressure: 140 (high risk factor - hypertension)
- Cholesterol: 289 (very high risk factor - hyperlipidemia)
- Fasting Blood Sugar: 0 (normal glucose level)
- Resting ECG: Normal (no abnormal findings)
- Maximum Heart Rate: 172 (high risk factor - age-related cardiovascular issues)
- Exercise Induced Angina: No (no risk factor)
- Old Peak Score: 0.0 (no abnormal findings)
- ST Slope: Up (no abnormal findings)

The patient is at very high risk for heart disease due to their elevated blood pressure, cholesterol levels, and age-related maximum heart rate. These factors significantly increase the likelihood of developing coronary artery disease or other cardiovascular conditions.
